In [0]:
# Imports
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI 
from langchain.prompts import ChatPromptTemplate 
import re 
import os 
from dotenv import load_dotenv 
import requests 

#load environment variables from .env
load_dotenv()

#read the api key for OpenWeather
api_key = os.getenv("OPENWEATHERMAP_API_KEY")
url = "https://api.openweathermap.org/data/2.5/forecast"

def fetch_weather_data(city: str) -> dict:    

    """Get a 5-day forecast for a city in a single call. 
      Args: 
      city : Name of the city passed as a string
      output: A dictionary with raw weather data.
    """   
    API_Call_Params = {"q":city, "appid":api_key, "units":"metric"} 

    API_Response = requests.get(url,params=API_Call_Params) 

  # API_Response_json = API_Response.json() 

    return API_Response 

def process_weather_data(data:dict) -> dict:
    """Converts raw weather data into a structured json.
    Parse the json to extract temperatures for each date.
    Returns a dictionary with forecasted temperatures for next 5 days.
    """ 

    if data.status_code == 200:
        API_Response_json = data.json()
    else:
        print ("API called with status : {API_Response.status_code}")

    weather_forecasts = {}

    for entry in API_Response_json["list"]:
        date = entry["dt_txt"].split()[0]
        temp = entry["main"]["temp"]
        weather_forecasts.setdefault(date,[]).append(temp)

    return weather_forecasts

def weather_advisory(weather_forecasts : dict) -> list:
    """Generate travel advisory after checking the maximum temperature.
    Format the output string.
    Return a list containing the date, maximum temperature and the advisory.
""" 
    advisory = []
    for date, temps in weather_forecasts.items():
        max_temp = max(temps)
        if max_temp > 40:
            advice = "Extreme Heat: Do not Travel"
        else:
            advice = "Safe to Travel"
            advisory.append(f"{date}: {max_temp:.1f}.{advice}") 

    return advisory 


@tool # To register the python function as an agentic tool. Returns a tool object.
def get_weather(city : str) -> str:
    """Call and execute all the helper functions.
    Args:
    City : The name of the city, for which forecast is requested, passed as a str.
    Output: A multi-line string of advisories.
    """
    #Function pipeline orchestration.    
    data = fetch_weather_data(city)
    forecasts = process_weather_data(data)
    advisories = weather_advisory(forecasts)
    
    return "\n".join (advisories) 

# Step 2 : Create LLM.

chat_model= ChatOpenAI(model="gpt-3.5-turbo",temperature =0)

# Step 3 : Prompt Template

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant"),
    ("user", "{input}"),
    ("placeholder","{agent_scratchpad}")
])

# Step 3: Create agent with model, tools, and system prompt

agent = create_tool_calling_agent(
    llm=chat_model,
    tools=[get_weather],
    prompt = prompt 
)

agent_executor = AgentExecutor(agent=agent,tools=[get_weather], verbose=False)

# Step 4: Invoke the agent
Agent_Output = agent_executor.invoke({
    "input" : "Give me the 5-day weather forecast for {city}, with daily max temperature and a travel advisory"
})
print (Agent_Output["output"])